# Phase 3 - NEURON to MuJoCo

Recommended workflow:
1. Choose whether this is a hemilineage video.
2. If yes, pick a `Hemi_XXX` folder and one of its run folders.
3. Filter the run's spikes to the added motor neurons for that hemilineage.
4. Map those motor neurons onto MuJoCo actuators.
5. Save outputs under `<save_tag>/<run_name>/` and render the video.

In [ ]:
from pathlib import Path
import json
import sys


def _dedupe_paths(paths):
    seen = set()
    out = []
    for path in paths:
        resolved = path.expanduser().resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        out.append(resolved)
    return out


def _find_phase3_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    extra = []
    for base in direct:
        extra.extend([base / 'Phase 3_WORKING', base / 'Phase 3'])
    for candidate in _dedupe_paths(direct + extra):
        if (candidate / 'src' / 'phase3_bridge').exists() and (candidate / 'configs' / 'phase3_video_profiles.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not locate the Phase 3 root. Start Jupyter from this repo or update the root detection cell.')


def _find_workspace_root(phase3_root):
    for candidate in [phase3_root.parent, *phase3_root.parents]:
        if (candidate / 'Phase 2').exists() or (candidate / 'Hemilineage Simulations').exists():
            return candidate
    return phase3_root.parent


PHASE3_ROOT = _find_phase3_root()
WORKSPACE_ROOT = _find_workspace_root(PHASE3_ROOT)
SRC_DIR = PHASE3_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import yaml
    from phase3_bridge.pipeline import (
        load_phase2_spike_times,
        load_phase2_motor_rates,
        load_phase2_timebase_ms,
        load_mapping_csv,
        summarize_mapping_coverage,
        summarize_rate_mapping_coverage,
        build_actuator_controls_from_spikes,
        build_actuator_controls_from_rates,
        filter_motor_rates_to_neuron_ids,
        save_controls_csv,
        plot_actuator_controls,
    )
    from phase3_bridge.hemilineage import (
        build_spike_mapping_summary,
        filter_spikes_to_neuron_ids,
        list_hemilineage_folders,
        list_run_folders,
        load_added_motor_neuron_ids,
        resolve_hemilineage_dir,
        summarize_focus_neuron_overlap,
    )
    from phase3_bridge.video_pipeline import apply_profile_transforms, list_model_cameras, render_camera_previews_mujoco, render_controls_mujoco
    from phase3_bridge.gait_audit import run_gait_expectation_audit
    from phase3_bridge.gait_compare import compare_gait_to_expected
    from phase3_bridge.expected_gait import DEFAULT_SEGMENT_OFFSETS_MS, render_expected_gait_video
except ModuleNotFoundError as exc:
    req_path = PHASE3_ROOT / 'requirements.txt'
    raise ModuleNotFoundError(
        f"Missing dependency {exc.name!r}. Install the Phase 3 requirements first, for example: pip install -r {req_path}"
    ) from exc

print('PHASE3_ROOT   =', PHASE3_ROOT)
print('WORKSPACE_ROOT=', WORKSPACE_ROOT)
print('requirements  =', PHASE3_ROOT / 'requirements.txt')

In [ ]:
HEMILINEAGE_SIM_ROOT = (WORKSPACE_ROOT / 'Hemilineage Simulations').resolve()
PHASE2_RUNS_ROOT = (WORKSPACE_ROOT / 'Phase 2' / 'data' / 'export_swc' / 'hemi_runs').resolve()

# Preset this notebook to the latest VIP glia run without DLM gap output.
# Set USE_PRESET_RUN = False to go back to the original interactive prompt flow.
USE_PRESET_RUN = True
PRESET_HEMILINEAGE_VIDEO = False
PRESET_HEMI_TAG = None
PRESET_RUN_NAME = 'gap_proof_gap_sustained_dep_ohmic_DLMnogap_glia_off'
PRESET_PHASE2_RUN_DIR = Path(
    '/Users/juanlopez2016/Desktop/Digifly_NEW/VIP_Glia_Sim/notebooks/debug/runs/gap_proof_gap_sustained_dep_ohmic_DLMnogap_glia_off'
).resolve()
PRESET_ADDED_MOTOR_IDS_CSV = None
PRESET_SAVE_TAG = 'VIP_glia_DLMnogap_v1'

if USE_PRESET_RUN:
    HEMILINEAGE_VIDEO = bool(PRESET_HEMILINEAGE_VIDEO)
    HEMI_TAG = PRESET_HEMI_TAG
    HEMI_DIR = resolve_hemilineage_dir(HEMILINEAGE_SIM_ROOT, HEMI_TAG) if HEMI_TAG else None
    available_runs = [PRESET_RUN_NAME]
    RUN_NAME = PRESET_RUN_NAME
    PHASE2_RUN_DIR = PRESET_PHASE2_RUN_DIR
    ADDED_MOTOR_IDS_CSV = PRESET_ADDED_MOTOR_IDS_CSV
    default_save_tag = PRESET_SAVE_TAG
else:
    hemilineage_reply = input('Generate a hemilineage video? [Y/n]: ').strip().lower()
    HEMILINEAGE_VIDEO = hemilineage_reply not in {'n', 'no'}

    if HEMILINEAGE_VIDEO:
        available_hemis = list_hemilineage_folders(HEMILINEAGE_SIM_ROOT)
        default_hemi = 'Hemi_09A' if 'Hemi_09A' in available_hemis else (available_hemis[0] if available_hemis else '')
        hemi_reply = input(f'Which hemi folder should Phase 3 use? [{default_hemi}]: ').strip()
        HEMI_TAG = hemi_reply or default_hemi
        HEMI_DIR = resolve_hemilineage_dir(HEMILINEAGE_SIM_ROOT, HEMI_TAG)

        available_runs = list_run_folders(HEMI_DIR)
        preferred_runs = ['hemi_09a_baseline', 'hemi_09a_baseline_1000ms-passive-posts']
        default_run = next((name for name in preferred_runs if name in available_runs), available_runs[0] if available_runs else '')
        run_reply = input(f'Which run folder inside {HEMI_TAG}? [{default_run}]: ').strip()
        RUN_NAME = run_reply or default_run
        if not RUN_NAME:
            raise ValueError(f'No run folders found inside {HEMI_DIR / "runs"}')

        PHASE2_RUN_DIR = (HEMI_DIR / 'runs' / RUN_NAME).resolve()
        ADDED_MOTOR_IDS_CSV = (HEMI_DIR / 'metadata' / 'added_motor_neuron_ids.csv').resolve()
        default_save_tag = HEMI_TAG
    else:
        HEMI_TAG = None
        HEMI_DIR = None
        available_runs = sorted(path.name for path in PHASE2_RUNS_ROOT.iterdir() if path.is_dir()) if PHASE2_RUNS_ROOT.exists() else []
        default_run = 'run_debug_full' if 'run_debug_full' in available_runs else (available_runs[0] if available_runs else '')
        run_reply = input(f'Which Phase 2 run folder should Phase 3 use? [{default_run}]: ').strip()
        RUN_NAME = run_reply or default_run
        if not RUN_NAME:
            raise ValueError(f'No Phase 2 run folders found inside {PHASE2_RUNS_ROOT}')

        PHASE2_RUN_DIR = (PHASE2_RUNS_ROOT / RUN_NAME).resolve()
        ADDED_MOTOR_IDS_CSV = None
        default_save_tag = RUN_NAME

    save_tag_reply = input(f'Save tag [{default_save_tag}]: ').strip()
    SAVE_TAG = save_tag_reply or default_save_tag

if USE_PRESET_RUN:
    SAVE_TAG = PRESET_SAVE_TAG

RUN_TAG = f'{SAVE_TAG}_{RUN_NAME}' if SAVE_TAG != RUN_NAME else RUN_NAME

MAPPING_CANDIDATES = [
    PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping_rebuilt.csv',
    PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping_with_groups.csv',
    PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping_updated.csv',
    PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping.csv',
]
MAPPING_CSV = next((path.resolve() for path in MAPPING_CANDIDATES if path.exists()), None)
if MAPPING_CSV is None:
    raise FileNotFoundError(f'No mapping CSV found. Checked: {[str(path) for path in MAPPING_CANDIDATES]}')

PROFILE_YAML = (PHASE3_ROOT / 'configs' / 'phase3_video_profiles.yaml').resolve()

# Spike -> activation kernel / rate normalization.
TAU_RISE_MS = 1.0
TAU_DECAY_MS = 6.0
GLOBAL_SCALE = 1.0
RATE_NORM_HZ = 200.0

# Profile / render controls
PROFILE_NAME = None  # Set a single profile name here if you want just one render.
PROFILE_NAMES = ['glia_jump_wingbeat_preview_femur20', 'glia_jump_wingbeat_preview_femur24', 'glia_jump_no_wing_preview_femur20', 'glia_jump_no_wing_preview_femur24']  # Batch-render both stable glia jump variants and their no-wing counterparts.
RENDER_ENABLED = True
CAMERA_PREVIEW_ENABLED = True
CAMERA_PREVIEW_TIME_MS = 410.0  # Show the glia burst shortly after the ~333 ms settle window.
CAMERA_PREVIEW_WIDTH = 480
CAMERA_PREVIEW_HEIGHT = 300
CAMERA_PREVIEW_NAMES = None  # None = preview all available cameras.

# Optional synthetic comparison render
GENERATE_EXPECTED_GAIT_COMPARE = False
EXPECTED_GAIT_FOCUS_MODE = 'active'
EXPECTED_GAIT_PROFILE_NAME = 'expected_gait_compare'
EXPECTED_GAIT_DURATION_MS = 18000.0
EXPECTED_GAIT_DT_MS = 5.0
EXPECTED_GAIT_STRIDE_PERIOD_MS = 1200.0
EXPECTED_GAIT_SWING_FRACTION = 0.32
EXPECTED_GAIT_SEGMENT_OFFSETS_MS = dict(DEFAULT_SEGMENT_OFFSETS_MS)

# Optional gait-likeness analysis
GENERATE_GAIT_COMPARE_REPORT = False

# MuJoCo world XML search order
WORLD_XML_CANDIDATES = [
    PHASE3_ROOT / 'data' / 'inputs' / 'flybody' / 'floor.xml',
    Path.home() / 'Desktop' / 'Digifly' / 'flybody-main' / 'flybody' / 'fruitfly' / 'assets' / 'floor.xml',
]
WORLD_XML = next((path.expanduser().resolve() for path in WORLD_XML_CANDIDATES if path.expanduser().exists()), None)

DERIVED_DIR = (PHASE3_ROOT / 'data' / 'derived' / SAVE_TAG / RUN_NAME).resolve()
FIG_DIR = (PHASE3_ROOT / 'data' / 'outputs' / 'figures' / SAVE_TAG / RUN_NAME).resolve()
VID_DIR = (PHASE3_ROOT / 'data' / 'outputs' / 'videos' / SAVE_TAG / RUN_NAME).resolve()

print('HEMILINEAGE_VIDEO =', HEMILINEAGE_VIDEO)
print('HEMI_TAG          =', HEMI_TAG)
print('RUN_NAME          =', RUN_NAME)
print('PHASE2_RUN_DIR    =', PHASE2_RUN_DIR)
print('ADDED_MOTOR_IDS   =', ADDED_MOTOR_IDS_CSV)
print('SAVE_TAG          =', SAVE_TAG)
print('MAPPING_CSV       =', MAPPING_CSV)
print('PROFILE_YAML      =', PROFILE_YAML)
print('WORLD_XML         =', WORLD_XML)
print('available_runs    =', available_runs)
print('EXPECTED_GAIT_COMPARE =', GENERATE_EXPECTED_GAIT_COMPARE)
if GENERATE_EXPECTED_GAIT_COMPARE:
    print('EXPECTED_GAIT_FOCUS_MODE =', EXPECTED_GAIT_FOCUS_MODE)
    print('EXPECTED_GAIT_PROFILE    =', EXPECTED_GAIT_PROFILE_NAME)
print('GAIT_COMPARE_REPORT =', GENERATE_GAIT_COMPARE_REPORT)
print('PROFILE_NAME =', PROFILE_NAME)
print('PROFILE_NAMES =', PROFILE_NAMES)
print('CAMERA_PREVIEW_ENABLED =', CAMERA_PREVIEW_ENABLED)
print('CAMERA_PREVIEW_TIME_MS =', CAMERA_PREVIEW_TIME_MS)
print('CAMERA_PREVIEW_NAMES =', CAMERA_PREVIEW_NAMES)


In [ ]:
with open(PROFILE_YAML, 'r', encoding='utf-8') as f:
    profile_doc = yaml.safe_load(f)

profiles = profile_doc.get('profiles', {})

_raw_profile_names = PROFILE_NAMES if PROFILE_NAMES is not None else PROFILE_NAME
if _raw_profile_names is None:
    PROFILE_NAMES_RESOLVED = [str(profile_doc.get('default_profile', 'biological_18s_preview'))]
elif isinstance(_raw_profile_names, (list, tuple, set)):
    PROFILE_NAMES_RESOLVED = [str(name) for name in _raw_profile_names]
else:
    PROFILE_NAMES_RESOLVED = [str(_raw_profile_names)]

missing_profiles = [name for name in PROFILE_NAMES_RESOLVED if name not in profiles]
if missing_profiles:
    raise KeyError(f'Profile(s) not found: {missing_profiles}. Available: {sorted(profiles)}')

PROFILE_DOCS = {name: profiles[name] for name in PROFILE_NAMES_RESOLVED}
PROFILE_NAME = PROFILE_NAMES_RESOLVED[0]
PROFILE = PROFILE_DOCS[PROFILE_NAME]

print('PROFILE_NAMES =', PROFILE_NAMES_RESOLVED)
for _profile_name in PROFILE_NAMES_RESOLVED:
    print(f'\\nPROFILE_NAME = {_profile_name}')
    print(json.dumps(PROFILE_DOCS[_profile_name], indent=2))

In [ ]:
config_json = PHASE2_RUN_DIR / 'config.json'
spike_csv = PHASE2_RUN_DIR / 'spike_times.csv'
motor_rates_csv = PHASE2_RUN_DIR / 'motor_rates.csv'

required_phase2_files = [config_json]
if spike_csv.exists():
    SOURCE_SIGNAL_KIND = 'spikes'
    required_phase2_files.append(spike_csv)
elif motor_rates_csv.exists():
    SOURCE_SIGNAL_KIND = 'motor_rates'
    required_phase2_files.append(motor_rates_csv)
else:
    raise FileNotFoundError(
        'Preset Phase 2 run is missing both spike_times.csv and motor_rates.csv: '
        + ', '.join(str(path) for path in [spike_csv, motor_rates_csv])
    )

missing_phase2_files = [path for path in required_phase2_files if not path.exists()]
if missing_phase2_files:
    raise FileNotFoundError(
        'Preset Phase 2 run is not finished yet. Missing required files: ' + ', '.join(str(path) for path in missing_phase2_files)
    )

mapping = load_mapping_csv(MAPPING_CSV)
mapping_basis_counts = mapping['mapping_basis'].fillna('missing').value_counts().to_dict() if 'mapping_basis' in mapping.columns else {}
mapping_confidence_counts = mapping['confidence'].fillna('missing').value_counts().to_dict() if 'confidence' in mapping.columns else {}

print('[mapping]')
print('  csv:', MAPPING_CSV)
for key, value in mapping_basis_counts.items():
    print(f'  mapping_basis[{key}] = {value}')
for key, value in mapping_confidence_counts.items():
    print(f'  confidence[{key}] = {value}')
print('  source_signal_kind:', SOURCE_SIGNAL_KIND)
print()

DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
VID_DIR.mkdir(parents=True, exist_ok=True)

added_motor_ids = None
motor_focus_stats = {}
motor_signal_csv = None
motor_activity_summary_csv = None

if ADDED_MOTOR_IDS_CSV is not None:
    added_motor_ids = load_added_motor_neuron_ids(ADDED_MOTOR_IDS_CSV)

if SOURCE_SIGNAL_KIND == 'spikes':
    spikes_all = load_phase2_spike_times(PHASE2_RUN_DIR)
    t_ms = load_phase2_timebase_ms(PHASE2_RUN_DIR)
    if added_motor_ids is not None:
        motor_focus_stats = summarize_focus_neuron_overlap(spikes_all, mapping, added_motor_ids)
        spikes = filter_spikes_to_neuron_ids(spikes_all, added_motor_ids)
        motor_signal_csv = DERIVED_DIR / f'{RUN_NAME}_added_motor_spikes.csv'
        spikes.to_csv(motor_signal_csv, index=False)
    else:
        spikes = spikes_all
    motor_activity_summary_df = build_spike_mapping_summary(spikes, mapping)
    motor_activity_summary_csv = DERIVED_DIR / f'{RUN_NAME}_added_motor_spike_summary.csv'
    motor_activity_summary_df.to_csv(motor_activity_summary_csv, index=False)
    coverage = summarize_mapping_coverage(spikes, mapping)
    controls_base_df, build_stats = build_actuator_controls_from_spikes(
        spikes=spikes,
        mapping=mapping,
        t_ms=t_ms,
        tau_rise_ms=TAU_RISE_MS,
        tau_decay_ms=TAU_DECAY_MS,
        scale=GLOBAL_SCALE,
    )
else:
    motor_rates_all = load_phase2_motor_rates(PHASE2_RUN_DIR)
    if added_motor_ids is not None:
        rate_ids_all = {int(str(col).split('_')[0]) for col in motor_rates_all.columns if col.endswith('_rate_hz')}
        motor_focus_stats = {
            'focus_motor_count': int(len(added_motor_ids)),
            'focus_rates_available_count': int(len(rate_ids_all.intersection(set(added_motor_ids)))),
            'focus_rates_missing_count': int(len(set(added_motor_ids) - rate_ids_all)),
        }
        motor_rates = filter_motor_rates_to_neuron_ids(motor_rates_all, added_motor_ids)
    else:
        motor_rates = motor_rates_all
    motor_signal_csv = DERIVED_DIR / f'{RUN_NAME}_added_motor_rates.csv'
    motor_rates.to_csv(motor_signal_csv, index=False)

    mapping_by_id = {int(nid): rows.copy() for nid, rows in mapping.groupby('mn_id')}
    summary_rows = []
    for col in motor_rates.columns:
        if col == 't_ms' or not str(col).endswith('_rate_hz'):
            continue
        nid = int(str(col).split('_')[0])
        vals = pd.to_numeric(motor_rates[col], errors='coerce').fillna(0.0).to_numpy(dtype=float)
        rows = mapping_by_id.get(nid)
        actuator_names = sorted(set(rows['actuator_name'])) if rows is not None else []
        summary_rows.append({
            'neuron_id': nid,
            'peak_rate_hz': float(np.nanmax(vals)) if vals.size else 0.0,
            'mean_rate_hz': float(np.nanmean(vals)) if vals.size else 0.0,
            'actuator_names': '; '.join(actuator_names),
            'mapping_rows': int(0 if rows is None else rows.shape[0]),
        })
    motor_activity_summary_df = pd.DataFrame(summary_rows).sort_values(['peak_rate_hz', 'neuron_id'], ascending=[False, True]).reset_index(drop=True)
    motor_activity_summary_csv = DERIVED_DIR / f'{RUN_NAME}_added_motor_rate_summary.csv'
    motor_activity_summary_df.to_csv(motor_activity_summary_csv, index=False)
    coverage = summarize_rate_mapping_coverage(motor_rates, mapping)
    controls_base_df, build_stats = build_actuator_controls_from_rates(
        rate_df=motor_rates,
        mapping=mapping,
        scale=GLOBAL_SCALE,
        rate_norm_hz=RATE_NORM_HZ,
    )

print('[coverage used for bridge]')
for key, value in coverage.items():
    print(f'  {key}: {value}')

if motor_focus_stats:
    print('\n[added motor overlap against the full run]')
    for key, value in motor_focus_stats.items():
        print(f'  {key}: {value}')
    print('[saved] motor-only signal  ->', motor_signal_csv)
    print('[saved] motor activity sum ->', motor_activity_summary_csv)

base_act = {column: controls_base_df[column].to_numpy(dtype=float) for column in controls_base_df.columns if column != 't_ms'}
PROFILE_RESULTS = {}

for _profile_name in PROFILE_NAMES_RESOLVED:
    _profile = PROFILE_DOCS[_profile_name]
    signal_cfg = _profile.get('signal', {})
    control_map_cfg = _profile.get('control_map', {})
    render_cfg = _profile.get('render', {})
    t_mod, mod_act = apply_profile_transforms(
        controls_base_df['t_ms'].to_numpy(dtype=float),
        base_act,
        signal_cfg,
    )

    controls_mod_df = pd.DataFrame({'t_ms': t_mod})
    for key in sorted(mod_act):
        controls_mod_df[key] = mod_act[key]

    base_csv = save_controls_csv(controls_base_df, DERIVED_DIR / f'controls_base_{_profile_name}.csv')
    mod_csv = save_controls_csv(controls_mod_df, DERIVED_DIR / f'controls_profiled_{_profile_name}.csv')
    base_fig = plot_actuator_controls(controls_base_df, FIG_DIR / f'{RUN_TAG}_{_profile_name}_base_top12.pdf', top_n=12)
    mod_fig = plot_actuator_controls(controls_mod_df, FIG_DIR / f'{RUN_TAG}_{_profile_name}_profiled_top12.pdf', top_n=12)

    summary = {
        'source_mode': 'hemilineage' if HEMILINEAGE_VIDEO else 'phase2_run',
        'phase2_signal_kind': SOURCE_SIGNAL_KIND,
        'hemi_tag': HEMI_TAG,
        'run_name': RUN_NAME,
        'save_tag': SAVE_TAG,
        'profile_name': _profile_name,
        'mapping_csv': str(MAPPING_CSV),
        'mapping_basis_counts': mapping_basis_counts,
        'mapping_confidence_counts': mapping_confidence_counts,
        'phase2_run_dir': str(PHASE2_RUN_DIR),
        'added_motor_ids_csv': str(ADDED_MOTOR_IDS_CSV) if ADDED_MOTOR_IDS_CSV else None,
        'world_xml': str(WORLD_XML) if WORLD_XML else None,
        'signal_profile': signal_cfg,
        'control_map': control_map_cfg,
        'render_profile': render_cfg,
        **coverage,
        **motor_focus_stats,
        **build_stats,
        'motor_only_signal_csv': str(motor_signal_csv) if motor_signal_csv else None,
        'motor_activity_summary_csv': str(motor_activity_summary_csv) if motor_activity_summary_csv else None,
        'base_controls_csv': str(base_csv),
        'profiled_controls_csv': str(mod_csv),
        'base_plot': str(base_fig),
        'profiled_plot': str(mod_fig),
    }

    summary_json = DERIVED_DIR / f'bridge_summary_{_profile_name}.json'
    with open(summary_json, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    PROFILE_RESULTS[_profile_name] = {
        'profile': _profile,
        'controls_mod_df': controls_mod_df,
        'summary': summary,
        'summary_json': summary_json,
        'base_csv': base_csv,
        'mod_csv': mod_csv,
        'base_fig': base_fig,
        'mod_fig': mod_fig,
    }

    print(f'[save:{_profile_name}] base controls     ->', base_csv)
    print(f'[save:{_profile_name}] profiled controls ->', mod_csv)
    print(f'[save:{_profile_name}] base figure       ->', base_fig)
    print(f'[save:{_profile_name}] profiled figure   ->', mod_fig)
    print(f'[save:{_profile_name}] summary           ->', summary_json)

_last_profile_name = PROFILE_NAMES_RESOLVED[-1]
PROFILE_NAME = _last_profile_name
PROFILE = PROFILE_DOCS[_last_profile_name]
controls_mod_df = PROFILE_RESULTS[_last_profile_name]['controls_mod_df']
summary_json = PROFILE_RESULTS[_last_profile_name]['summary_json']
print('[save] completed profiles ->', PROFILE_NAMES_RESOLVED)

motor_activity_summary_df.head(20)


In [ ]:
render_report_path = DERIVED_DIR / f'render_report_{PROFILE_NAME}.json'

render_cfg = PROFILE.get('render', {})
control_map_cfg = PROFILE.get('control_map', {})

if WORLD_XML is not None and CAMERA_PREVIEW_ENABLED:
    try:
        camera_catalog = list_model_cameras(WORLD_XML)
        available_camera_names = [entry['name'] for entry in camera_catalog if entry.get('name')]
        preview_names = list(CAMERA_PREVIEW_NAMES) if CAMERA_PREVIEW_NAMES else available_camera_names
        preview_frames = render_camera_previews_mujoco(
            mjcf_xml=WORLD_XML,
            t_ms=controls_mod_df['t_ms'].to_numpy(dtype=float),
            act_controls={column: controls_mod_df[column].to_numpy(dtype=float) for column in controls_mod_df.columns if column != 't_ms'},
            camera_names=preview_names,
            preview_time_ms=float(CAMERA_PREVIEW_TIME_MS),
            camera_distance_factor=float(render_cfg.get('camera_distance_factor', 8.0)),
            camera_fovy_deg=float(render_cfg.get('camera_fovy_deg', 75.0)),
            width=int(CAMERA_PREVIEW_WIDTH),
            height=int(CAMERA_PREVIEW_HEIGHT),
            target_ctrl_fraction=float(control_map_cfg.get('target_ctrl_fraction', 0.7)),
            percentile=float(control_map_cfg.get('percentile', 95.0)),
            bias_to_mid=bool(control_map_cfg.get('bias_to_mid', True)),
            clip=bool(control_map_cfg.get('clip', True)),
            output_gains=dict(control_map_cfg.get('output_gains', {}) or {}),
            actuator_boosts=list(render_cfg.get('actuator_boosts', []) or []),
        )
        if preview_frames:
            cols = 2 if len(preview_names) <= 4 else 3
            rows = int(np.ceil(len(preview_names) / cols))
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.8, rows * 3.4))
            axes = np.atleast_1d(axes).ravel()
            for ax, name in zip(axes, preview_names):
                frame = preview_frames.get(name)
                ax.axis('off')
                if frame is None:
                    ax.set_title(f'{name} (unavailable)')
                    continue
                ax.imshow(frame)
                ax.set_title(name)
            for ax in axes[len(preview_names):]:
                ax.axis('off')
            fig.suptitle(f'Camera previews at {float(CAMERA_PREVIEW_TIME_MS):.1f} ms', fontsize=14)
            fig.tight_layout()
            plt.show()
            print('[render] available cameras ->', available_camera_names)
            print('[render] preview cameras   ->', preview_names)
    except Exception as preview_exc:
        print('[render] camera previews unavailable:', preview_exc)

if not RENDER_ENABLED:
    print('[render] skipped (RENDER_ENABLED=False)')
elif WORLD_XML is None:
    print('[render] skipped (no WORLD_XML found)')
    print('[render] add floor.xml under', PHASE3_ROOT / 'data' / 'inputs' / 'flybody')
else:
    video_out = VID_DIR / f'{RUN_TAG}_{PROFILE_NAME}.mp4'
    try:
        render_report = render_controls_mujoco(
            mjcf_xml=WORLD_XML,
            t_ms=controls_mod_df['t_ms'].to_numpy(dtype=float),
            act_controls={column: controls_mod_df[column].to_numpy(dtype=float) for column in controls_mod_df.columns if column != 't_ms'},
            out_video=video_out,
            camera_name=str(render_cfg.get('camera_name', 'track2')),
            camera_distance_factor=float(render_cfg.get('camera_distance_factor', 8.0)),
            camera_fovy_deg=float(render_cfg.get('camera_fovy_deg', 75.0)),
            render_hz=int(render_cfg.get('render_hz', 120)),
            slowmo=float(render_cfg.get('slowmo', 20.0)),
            width=int(render_cfg.get('width', 1280)),
            height=int(render_cfg.get('height', 720)),
            target_ctrl_fraction=float(control_map_cfg.get('target_ctrl_fraction', 0.7)),
            percentile=float(control_map_cfg.get('percentile', 95.0)),
            bias_to_mid=bool(control_map_cfg.get('bias_to_mid', True)),
            clip=bool(control_map_cfg.get('clip', True)),
            output_gains=dict(control_map_cfg.get('output_gains', {}) or {}),
            actuator_boosts=list(render_cfg.get('actuator_boosts', []) or []),
        )
        with open(render_report_path, 'w', encoding='utf-8') as f:
            json.dump(render_report, f, indent=2)
        print('[render] video  ->', video_out)
        print('[render] report ->', render_report_path)
        print('[render] stats  ->', json.dumps(render_report, indent=2))
        try:
            from IPython.display import Video, display
            display(Video(filename=str(video_out), embed=True, html_attributes='controls'))
        except Exception as preview_exc:
            print('[render] inline preview unavailable:', preview_exc)
    except Exception as exc:
        err = {'error': str(exc)}
        with open(render_report_path, 'w', encoding='utf-8') as f:
            json.dump(err, f, indent=2)
        print('[render] failed:', exc)
        print('[render] wrote error report ->', render_report_path)


In [ ]:
if not RENDER_ENABLED:
    print('[render-secondary] skipped (RENDER_ENABLED=False)')
elif WORLD_XML is None:
    print('[render-secondary] skipped (no WORLD_XML found)')
elif 'PROFILE_RESULTS' not in globals() or not PROFILE_RESULTS:
    print('[render-secondary] skipped (run the build cell first)')
else:
    secondary_profile_names = [name for name in PROFILE_NAMES_RESOLVED if name != PROFILE_NAME]
    if not secondary_profile_names:
        print('[render-secondary] no secondary profiles to render')
    for _profile_name in secondary_profile_names:
        _profile = PROFILE_DOCS[_profile_name]
        _render_cfg = _profile.get('render', {})
        _control_map_cfg = _profile.get('control_map', {})
        _controls_mod_df = PROFILE_RESULTS[_profile_name]['controls_mod_df']
        _render_report_path = DERIVED_DIR / f'render_report_{_profile_name}.json'
        _video_out = VID_DIR / f'{RUN_TAG}_{_profile_name}.mp4'
        try:
            _render_report = render_controls_mujoco(
                mjcf_xml=WORLD_XML,
                t_ms=_controls_mod_df['t_ms'].to_numpy(dtype=float),
                act_controls={column: _controls_mod_df[column].to_numpy(dtype=float) for column in _controls_mod_df.columns if column != 't_ms'},
                out_video=_video_out,
                camera_name=str(_render_cfg.get('camera_name', 'track2')),
                camera_distance_factor=float(_render_cfg.get('camera_distance_factor', 8.0)),
                camera_fovy_deg=float(_render_cfg.get('camera_fovy_deg', 75.0)),
                render_hz=int(_render_cfg.get('render_hz', 120)),
                slowmo=float(_render_cfg.get('slowmo', 20.0)),
                width=int(_render_cfg.get('width', 1280)),
                height=int(_render_cfg.get('height', 720)),
                target_ctrl_fraction=float(_control_map_cfg.get('target_ctrl_fraction', 0.7)),
                percentile=float(_control_map_cfg.get('percentile', 95.0)),
                bias_to_mid=bool(_control_map_cfg.get('bias_to_mid', True)),
                clip=bool(_control_map_cfg.get('clip', True)),
                output_gains=dict(_control_map_cfg.get('output_gains', {}) or {}),
                actuator_boosts=list(_render_cfg.get('actuator_boosts', []) or []),
            )
            with open(_render_report_path, 'w', encoding='utf-8') as f:
                json.dump(_render_report, f, indent=2)
            print(f'[render-secondary:{_profile_name}] video  ->', _video_out)
            print(f'[render-secondary:{_profile_name}] report ->', _render_report_path)
            print(f'[render-secondary:{_profile_name}] stats  ->', json.dumps(_render_report, indent=2))
        except Exception as exc:
            _err = {'error': str(exc)}
            with open(_render_report_path, 'w', encoding='utf-8') as f:
                json.dump(_err, f, indent=2)
            print(f'[render-secondary:{_profile_name}] failed:', exc)
            print(f'[render-secondary:{_profile_name}] wrote error report ->', _render_report_path)


In [ ]:
# EXPECTED_GAIT_DIR = (DERIVED_DIR / 'expected_gait').resolve()
# expected_gait_summary_path = EXPECTED_GAIT_DIR / f'expected_gait_summary_{EXPECTED_GAIT_FOCUS_MODE}_{EXPECTED_GAIT_PROFILE_NAME}.json'

# if not GENERATE_EXPECTED_GAIT_COMPARE:
#     print('[expected gait] skipped (GENERATE_EXPECTED_GAIT_COMPARE=False)')
# else:
#     try:
#         expected_gait_summary = render_expected_gait_video(
#             phase3_root=PHASE3_ROOT,
#             mapping_csv=MAPPING_CSV,
#             out_dir=EXPECTED_GAIT_DIR,
#             run_dir=PHASE2_RUN_DIR if PHASE2_RUN_DIR.exists() else None,
#             added_motor_ids_csv=(ADDED_MOTOR_IDS_CSV if ADDED_MOTOR_IDS_CSV and ADDED_MOTOR_IDS_CSV.exists() else None),
#             save_tag=SAVE_TAG,
#             run_name=RUN_NAME,
#             profile_doc=profile_doc,
#             profile_name=EXPECTED_GAIT_PROFILE_NAME,
#             focus_mode=EXPECTED_GAIT_FOCUS_MODE,
#             duration_ms=float(EXPECTED_GAIT_DURATION_MS),
#             dt_ms=float(EXPECTED_GAIT_DT_MS),
#             stride_period_ms=float(EXPECTED_GAIT_STRIDE_PERIOD_MS),
#             swing_fraction=float(EXPECTED_GAIT_SWING_FRACTION),
#             segment_offsets_ms=EXPECTED_GAIT_SEGMENT_OFFSETS_MS,
#         )
#         print('[expected gait] summary ->', expected_gait_summary_path)
#         print('[expected gait] phase   ->', expected_gait_summary.get('phase_channels_csv'))
#         print('[expected gait] weights ->', expected_gait_summary.get('weights_csv'))
#         print('[expected gait] video   ->', expected_gait_summary.get('video_out'))
#         print('[expected gait] report  ->', expected_gait_summary.get('render_report'))
#         if expected_gait_summary.get('render_stats'):
#             print('[expected gait] stats  ->', json.dumps(expected_gait_summary.get('render_stats', {}), indent=2))
#         render_error = (expected_gait_summary.get('render_stats') or {}).get('error')
#         if expected_gait_summary.get('video_out') and render_error is None:
#             try:
#                 from IPython.display import Video, display
#                 display(Video(filename=str(expected_gait_summary['video_out']), embed=True, html_attributes='controls'))
#             except Exception as preview_exc:
#                 print('[expected gait] inline preview unavailable:', preview_exc)
#         elif render_error:
#             print('[expected gait] render error:', render_error)
#     except Exception as exc:
#         print('[expected gait] failed:', exc)


In [ ]:
SIM_GAIT_AUDIT_DIR = (PHASE3_ROOT / 'data' / 'derived' / 'gait_expectation' / SAVE_TAG / RUN_NAME).resolve()
GAIT_COMPARE_DIR = (PHASE3_ROOT / 'data' / 'derived' / 'gait_compare' / SAVE_TAG / RUN_NAME).resolve()
EXPECTED_GAIT_DIR = (DERIVED_DIR / 'expected_gait').resolve()

if not GENERATE_GAIT_COMPARE_REPORT:
    print('[gait compare] skipped (GENERATE_GAIT_COMPARE_REPORT=False)')
elif SOURCE_SIGNAL_KIND != 'spikes':
    print('[gait compare] skipped for non-spike phase2 source:', SOURCE_SIGNAL_KIND)
else:
    gait_audit_report = run_gait_expectation_audit(
        run_dir=PHASE2_RUN_DIR,
        mapping_csv=MAPPING_CSV,
        out_dir=SIM_GAIT_AUDIT_DIR,
        tau_rise_ms=TAU_RISE_MS,
        tau_decay_ms=TAU_DECAY_MS,
    )
    print('[gait audit] report ->', SIM_GAIT_AUDIT_DIR / 'gait_expectation_report.json')
    print('[gait audit] non_gait_like ->', gait_audit_report.get('diagnosis', {}).get('activity_pattern_likely_non_gait'))
    if gait_audit_report.get('warnings'):
        print('[gait audit] warnings ->')
        for warning in gait_audit_report.get('warnings', []):
            print('  -', warning)

    expected_phase_csv = EXPECTED_GAIT_DIR / f'expected_gait_phase_channels_{EXPECTED_GAIT_FOCUS_MODE}.csv'
    expected_weight_csv = EXPECTED_GAIT_DIR / f'expected_gait_weights_{EXPECTED_GAIT_FOCUS_MODE}.csv'
    if expected_phase_csv.exists() and expected_weight_csv.exists():
        gait_compare_report = compare_gait_to_expected(
            sim_audit_dir=SIM_GAIT_AUDIT_DIR,
            expected_dir=EXPECTED_GAIT_DIR,
            out_dir=GAIT_COMPARE_DIR,
        )
        print('[gait compare] report ->', GAIT_COMPARE_DIR / 'gait_compare_report.json')
        print('[gait compare] closeness_score ->', gait_compare_report.get('overall_closeness_score'))
        print('[gait compare] classification  ->', gait_compare_report.get('overall_classification'))
        if gait_compare_report.get('recommendations'):
            print('[gait compare] recommendations ->')
            for item in gait_compare_report.get('recommendations', []):
                print('  -', item)
        pd.read_csv(GAIT_COMPARE_DIR / 'phase_distribution_comparison.csv').sort_values('norm_diff', ascending=False).head(12)
    else:
        print('[gait compare] skipped because expected gait artifacts are missing in', EXPECTED_GAIT_DIR)
        print('[gait compare] run the expected gait cell first or leave GENERATE_EXPECTED_GAIT_COMPARE=True')


## Notes

- In hemilineage mode the bridge uses only spikes from `added_motor_neuron_ids.csv`.
- Outputs are saved under `data/derived/<save_tag>/<run_name>/`, `data/outputs/figures/<save_tag>/<run_name>/`, and `data/outputs/videos/<save_tag>/<run_name>/`.
- The notebook can also render a synthetic comparison video under `.../<save_tag>/<run_name>/expected_gait/`; use `GENERATE_EXPECTED_GAIT_COMPARE` and `EXPECTED_GAIT_FOCUS_MODE` in the config cell.
- The notebook can also write a gait-distance report under `data/derived/gait_compare/<save_tag>/<run_name>/`; use `GENERATE_GAIT_COMPARE_REPORT` to control that analysis.
- Successful renders are also displayed inline in Jupyter when the notebook frontend supports embedded video.
- Leave the save-tag prompt blank to use the `Hemi_XXX` folder name by default.